# 🇿🇦 CashContant — Exploratory Data Analysis

**Author:** Sifiso Mnguni  
**Data Sources:** South African Reserve Bank (SARB), World Bank, SmartCash SA  
**Objective:** Analyse the persistent demand for physical cash in South Africa and quantify the associated security risks.

---

## Research Questions

1. Has cash in circulation (M0) grown or declined in real terms since 2010?
2. How has ATM density changed across South Africa?
3. What patterns exist in personal ATM withdrawal behaviour?
4. How prevalent are potentially fraudulent transactions in ATM data?

---

In [ ]:
# --- Imports ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_palette('Greens_r')

print('Libraries loaded ✅')

## 1. Cash in Circulation (M0) — SARB Data

M0 is the narrowest measure of money supply — it represents the total physical banknotes and coins in circulation. A growing M0 despite digital payment expansion suggests structural cash dependency.

In [ ]:
# Load M0 data
df_m0 = pd.read_csv('../data/cash_in_circulation.csv', skiprows=1)
df_m0 = df_m0[['Date', 'M0']].copy()
df_m0['M0'] = pd.to_numeric(df_m0['M0'].astype(str).str.replace(',', ''), errors='coerce')
df_m0['Date'] = pd.to_datetime(df_m0['Date'], format='%b, %Y', errors='coerce')
df_m0 = df_m0.dropna().sort_values('Date')

# YoY Growth
df_m0['YoY_Change'] = df_m0['M0'].pct_change(12) * 100

print(f'M0 data shape: {df_m0.shape}')
print(f'Date range: {df_m0.Date.min().date()} → {df_m0.Date.max().date()}')
df_m0.tail()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# M0 level
axes[0].plot(df_m0['Date'], df_m0['M0'] / 1e3, color='#2e7d52', linewidth=2.5)
axes[0].fill_between(df_m0['Date'], df_m0['M0'] / 1e3, alpha=0.15, color='#2e7d52')
axes[0].set_title('Cash in Circulation (M0)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('R Billions')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[0].axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-03-01'),
                alpha=0.1, color='red', label='COVID-19')
axes[0].legend()

# YoY Change
colors = ['#4caf50' if x >= 0 else '#e53935' for x in df_m0['YoY_Change'].dropna()]
axes[1].bar(df_m0['Date'].iloc[12:], df_m0['YoY_Change'].dropna(), color=colors, width=20)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('M0 Year-on-Year % Change', fontsize=13, fontweight='bold')
axes[1].set_ylabel('% Change')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('../images/m0_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n💡 Insight: M0 has grown consistently — physical cash demand is structural, not declining.')

## 2. ATM Density — World Bank Data

ATM density (ATMs per 100,000 adults) is a proxy for cash infrastructure. South Africa's trajectory tells us whether banks are expanding or contracting their cash distribution networks.

In [ ]:
# Load World Bank ATM density data
df_wb = pd.read_csv('../data/API_FB.ATM.TOTL.P5_DS2_en_csv_v2_6160.csv', skiprows=4)
df_sa = df_wb[df_wb['Country Name'] == 'South Africa'].copy()

years = [str(y) for y in range(2004, 2024)]
df_sa_long = df_sa.melt(id_vars=['Country Name'], value_vars=years,
                         var_name='Year', value_name='ATM_Density')
df_sa_long['Year'] = pd.to_datetime(df_sa_long['Year'], format='%Y')
df_sa_long['ATM_Density'] = pd.to_numeric(df_sa_long['ATM_Density'], errors='coerce')
df_sa_long = df_sa_long.dropna()

print(f'Peak ATM density: {df_sa_long.ATM_Density.max():.1f} ATMs/100k adults')
print(f'Latest ATM density: {df_sa_long.ATM_Density.iloc[-1]:.1f} ATMs/100k adults')
df_sa_long.tail()

In [ ]:
fig, ax = plt.subplots()
ax.plot(df_sa_long['Year'], df_sa_long['ATM_Density'],
        marker='o', markersize=5, color='#1a4731', linewidth=2.5)
ax.set_title('South Africa — ATM Density (2004–2023)', fontsize=13, fontweight='bold')
ax.set_ylabel('ATMs per 100,000 Adults')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('../images/atm_density.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Personal ATM Withdrawal Behaviour

Analysing individual ATM usage patterns: frequency, amounts, timing, and branch preferences.

In [ ]:
# Load personal ATM data
df_atm = pd.read_csv('../data/atm_withdrawals.csv', delimiter=';')
df_atm = df_atm.loc[:, ~df_atm.columns.str.startswith('Unnamed')]
df_atm['timestamp'] = pd.to_datetime(df_atm['timestamp'], errors='coerce')
df_atm = df_atm.dropna(subset=['timestamp'])
df_atm['month'] = df_atm['timestamp'].dt.to_period('M')
df_atm['hour'] = df_atm['timestamp'].dt.hour
df_atm['day_of_week'] = df_atm['timestamp'].dt.day_name()

print(f'Records: {len(df_atm):,}')
print(f'Date range: {df_atm.timestamp.min().date()} → {df_atm.timestamp.max().date()}')
print(f'Total withdrawn: R{df_atm.withdrawal_amount.sum():,.2f}')
df_atm.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Monthly totals
monthly = df_atm.groupby('month')['withdrawal_amount'].sum()
monthly.plot(kind='bar', ax=axes[0], color='#2e7d52', edgecolor='white')
axes[0].axhline(monthly.mean(), color='#c9a84c', linestyle='--', label=f'Avg: R{monthly.mean():,.0f}')
axes[0].set_title('Monthly Withdrawals', fontweight='bold')
axes[0].set_ylabel('R')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()

# Hour of day
hourly = df_atm['hour'].value_counts().sort_index()
colors_hr = ['#e53935' if h in range(0, 5) else '#2e7d52' for h in hourly.index]
axes[1].bar(hourly.index, hourly.values, color=colors_hr)
axes[1].set_title('Withdrawals by Hour of Day', fontweight='bold')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Count')
axes[1].axvspan(-0.5, 4.5, alpha=0.1, color='red', label='After-hours risk zone')
axes[1].legend()

# Amount distribution
df_atm['withdrawal_amount'].plot(kind='hist', bins=30, ax=axes[2],
                                   color='#2e7d52', edgecolor='white', alpha=0.85)
axes[2].set_title('Withdrawal Amount Distribution', fontweight='bold')
axes[2].set_xlabel('Amount (R)')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../images/withdrawal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Security & Fraud Risk Analysis

Running the `model_security` module to flag anomalous transactions and visualise the risk distribution.

In [ ]:
import sys
sys.path.append('..')
from models.model_security import security_analysis

# Rename column to match expected schema
scan_df = df_atm.copy()
if 'withdrawal_amount' in scan_df.columns:
    scan_df = scan_df.rename(columns={'withdrawal_amount': 'amount'})

summary, scored_df = security_analysis(scan_df)
print(summary)

In [ ]:
# Risk level distribution
risk_counts = scored_df['risk_level'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors_risk = ['#4caf50', '#f0d080', '#fb8c00', '#e53935']
risk_counts.plot(kind='pie', ax=axes[0], colors=colors_risk,
                  autopct='%1.1f%%', startangle=90, legend=False)
axes[0].set_title('Transaction Risk Distribution', fontweight='bold')
axes[0].set_ylabel('')

# Anomaly score histogram
scored_df['anomaly_score'].plot(kind='hist', bins=20, ax=axes[1],
                                 color='#2e7d52', edgecolor='white', alpha=0.85)
axes[1].axvline(60, color='#e53935', linestyle='--', label='High risk threshold')
axes[1].axvline(30, color='#fb8c00', linestyle='--', label='Medium risk threshold')
axes[1].set_title('Anomaly Score Distribution (0–100)', fontweight='bold')
axes[1].set_xlabel('Anomaly Score')
axes[1].set_ylabel('Transactions')
axes[1].legend()

plt.tight_layout()
plt.savefig('../images/risk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Summary of Findings

| Finding | Evidence |
|---|---|
| Cash demand is structural | M0 has grown steadily year-on-year |
| ATM network is maturing | ATM density peaked and stabilised, not declining |
| End-of-month spikes visible | Monthly withdrawal chart shows salary cycle effect |
| After-hours risk is quantifiable | X% of transactions occur in the 00h–04h risk window |
| Fraud detection is feasible | IQR + velocity + travel checks flag meaningful anomalies |

---

**Next steps:** Run the full CashContant Streamlit app (`streamlit run app.py`) to explore these findings interactively.